In [1]:
pip install timm

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# -----------------------------
# Config
# -----------------------------
ROOT = "../fiot_highway2-main"
TRAIN_TXT = os.path.join(ROOT, "train.txt")
TEST_TXT  = os.path.join(ROOT, "test.txt")

NUM_CLASSES = 9
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# keep it small for more speed
MAX_TRAIN_SAMPLES = 2000
MAX_TEST_SAMPLES  = 1000

DS_FREQ = 4   # 512 -> 128
DS_TIME = 4   # 243 -> 61

BATCH_SIZE = 32
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------------
# Data loading helpers
# -----------------------------
def load_txt_list(txt_path):
    entries = np.loadtxt(txt_path, dtype=str)
    if entries.ndim == 1:
        entries = entries.reshape(1, -1)

    pairs = []
    for rel_path, label_str in entries:
        full_path = os.path.join(ROOT, rel_path)
        pairs.append((full_path, int(label_str)))
    return pairs


def maybe_subsample(pairs, max_samples):
    if max_samples is None or max_samples >= len(pairs):
        return pairs
    idx = np.random.choice(len(pairs), size=max_samples, replace=False)
    return [pairs[i] for i in idx]


class PSDDataset(Dataset):
    """
    Loads PSD .npy arrays, per-sample normalize, downsample, then make ViT-friendly image tensor.
    Output:
      x: float32 tensor (3, 224, 224)
      y: int64 tensor ()
    """
    def __init__(self, pairs, ds_f=4, ds_t=4, out_size=224):
        self.pairs = pairs
        self.ds_f = ds_f
        self.ds_t = ds_t
        self.out_size = out_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        mat = np.load(path).astype(np.float32)  # (512,243)

        # per-sample normalization
        mat = (mat - mat.mean()) / (mat.std() + 1e-6)

        # downsample
        mat = mat[::self.ds_f, ::self.ds_t]  # e.g. (128,61)

        # make tensor (1,H,W)
        x = torch.from_numpy(mat).unsqueeze(0)  # (1,H,W)

        # resize to (1,224,224)
        x = F.interpolate(x.unsqueeze(0), size=(self.out_size, self.out_size),
                          mode="bilinear", align_corners=False).squeeze(0)  # (1,224,224)

        # repeat channels to (3,224,224) since most ViTs expect 3 channels
        x = x.repeat(3, 1, 1)

        y = torch.tensor(label, dtype=torch.long)
        return x, y


# -----------------------------
# Metrics
# -----------------------------
@torch.no_grad()
def eval_loader(model, loader, n_classes=9):
    model.eval()
    all_preds = []
    all_true = []

    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == y).sum().item()
        total += y.numel()

        all_preds.append(preds.cpu().numpy())
        all_true.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_true)

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
    per_class_recall = cm.diagonal() / np.clip(cm.sum(axis=1), 1, None)  # recall = "per-class accuracy"
    acc = correct / max(total, 1)

    return acc, per_class_recall, cm, y_true, y_pred


def class_weights_from_labels(y, n_classes=9):
    counts = np.bincount(y, minlength=n_classes).astype(np.float32)
    weights = counts.sum() / np.clip(counts, 1, None)
    weights = weights / weights.mean()  # normalize (optional)
    return torch.tensor(weights, dtype=torch.float32)


# -----------------------------
# Main
# -----------------------------
train_pairs = load_txt_list(TRAIN_TXT)
test_pairs  = load_txt_list(TEST_TXT)

train_pairs = maybe_subsample(train_pairs, MAX_TRAIN_SAMPLES)
test_pairs  = maybe_subsample(test_pairs,  MAX_TEST_SAMPLES)

# split train -> train/val stratified
labels = np.array([y for _, y in train_pairs])
idx_train, idx_val = train_test_split(
    np.arange(len(train_pairs)),
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=labels
)
train_split = [train_pairs[i] for i in idx_train]
val_split   = [train_pairs[i] for i in idx_val]

train_ds = PSDDataset(train_split, ds_f=DS_FREQ, ds_t=DS_TIME, out_size=224)
val_ds   = PSDDataset(val_split,   ds_f=DS_FREQ, ds_t=DS_TIME, out_size=224)
test_ds  = PSDDataset(test_pairs,  ds_f=DS_FREQ, ds_t=DS_TIME, out_size=224)

isAva = torch.cuda.is_available()

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=isAva)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=isAva)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=isAva)

# ViT model
model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

# loss with class weights (helps imbalance)
y_train_split = np.array([y for _, y in train_split])
cw = class_weights_from_labels(y_train_split, n_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=cw)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * y.size(0)

    train_loss = running_loss / len(train_ds)

    val_acc, val_per_class, val_cm, _, _ = eval_loader(model, val_loader, n_classes=NUM_CLASSES)

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f}")
    print("Val per-class accuracy (recall) %:", np.round(val_per_class * 100, 2).tolist())

    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# Load best model by val
if best_state is not None:
    model.load_state_dict(best_state)

test_acc, test_per_class, test_cm, y_true, y_pred = eval_loader(model, test_loader, n_classes=NUM_CLASSES)

print("\n=== TEST RESULTS ===")
print("Test accuracy:", round(test_acc, 4))
print("Per-class accuracy (recall) %:", np.round(test_per_class * 100, 2).tolist())

print("\nClassification report (TEST):")
print(classification_report(y_true, y_pred, digits=4))

print("\nConfusion matrix (TEST):\n", test_cm)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

ROOT = "../fiot_highway2-main"
TRAIN_TXT = os.path.join(ROOT, "train.txt")
TEST_TXT  = os.path.join(ROOT, "test.txt")

NUM_CLASSES = 9
RANDOM_SEED = 42

MAX_TRAIN_SAMPLES = 2000
MAX_TEST_SAMPLES  = 1000

DS_FREQ = 4
DS_TIME = 4

BATCH_SIZE = 32
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------------
# Data loading helpers
# -----------------------------
def load_txt_list(txt_path):
    entries = np.loadtxt(txt_path, dtype=str)
    if entries.ndim == 1:
        entries = entries.reshape(1, -1)

    pairs = []
    for rel_path, label_str in entries:
        full_path = os.path.join(ROOT, rel_path)
        pairs.append((full_path, int(label_str)))
    return pairs


def maybe_subsample(pairs, max_samples):
    if max_samples is None or max_samples >= len(pairs):
        return pairs
    idx = np.random.choice(len(pairs), size=max_samples, replace=False)
    return [pairs[i] for i in idx]


class PSDDataset(Dataset):
    """
    Loads PSD .npy arrays, per-sample normalize, downsample, then make ViT-friendly image tensor.
    Output:
      x: float32 tensor (3, 224, 224)
      y: int64 tensor ()
    """
    def __init__(self, pairs, ds_f=4, ds_t=4, out_size=224):
        self.pairs = pairs
        self.ds_f = ds_f
        self.ds_t = ds_t
        self.out_size = out_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        mat = np.load(path).astype(np.float32)  # (512,243)

        mat = (mat - mat.mean()) / (mat.std() + 1e-6)
        mat = mat[::self.ds_f, ::self.ds_t]  # (128,61)

        x = torch.from_numpy(mat).unsqueeze(0)  # (1,H,W)
        x = F.interpolate(
            x.unsqueeze(0),
            size=(self.out_size, self.out_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)  # (1,224,224)

        x = x.repeat(3, 1, 1)  # (3,224,224)
        y = torch.tensor(label, dtype=torch.long)
        return x, y


def class_weights_from_labels(y, n_classes=9):
    counts = np.bincount(y, minlength=n_classes).astype(np.float32)
    weights = counts.sum() / np.clip(counts, 1, None)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


# -----------------------------
# Lightning DataModule
# -----------------------------
class PSDDataModule(L.LightningDataModule):
    def __init__(self):
        super().__init__()
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def setup(self, stage=None):
        train_pairs = load_txt_list(TRAIN_TXT)
        test_pairs  = load_txt_list(TEST_TXT)

        train_pairs = maybe_subsample(train_pairs, MAX_TRAIN_SAMPLES)
        test_pairs  = maybe_subsample(test_pairs,  MAX_TEST_SAMPLES)

        labels = np.array([y for _, y in train_pairs])
        idx_train, idx_val = train_test_split(
            np.arange(len(train_pairs)),
            test_size=0.2,
            random_state=RANDOM_SEED,
            stratify=labels
        )
        train_split = [train_pairs[i] for i in idx_train]
        val_split   = [train_pairs[i] for i in idx_val]

        self.train_ds = PSDDataset(train_split, ds_f=DS_FREQ, ds_t=DS_TIME, out_size=224)
        self.val_ds   = PSDDataset(val_split,   ds_f=DS_FREQ, ds_t=DS_TIME, out_size=224)
        self.test_ds  = PSDDataset(test_pairs,  ds_f=DS_FREQ, ds_t=DS_TIME, out_size=224)

        # class weights from *train* labels
        y_train_split = np.array([y for _, y in train_split])
        self.class_weights = class_weights_from_labels(y_train_split, n_classes=NUM_CLASSES)

    def train_dataloader(self):
        return DataLoader(
            self.train_ds, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=0, pin_memory=torch.cuda.is_available()
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=0, pin_memory=torch.cuda.is_available()
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=0, pin_memory=torch.cuda.is_available()
        )


# -----------------------------
# LightningModule (ViT + metrics)
# -----------------------------
class LitViT(L.LightningModule):
    def __init__(
        self,
        model_name: str,
        num_classes: int,
        lr: float,
        weight_decay: float,
        scheduler: str = "cosine",  # "none" | "cosine" | "step" | "plateau"
        step_size: int = 10,
        gamma: float = 0.5,
        plateau_patience: int = 3,
        plateau_factor: float = 0.5,
        class_weights: torch.Tensor | None = None,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["class_weights"])

        self.model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)

        cw = class_weights if class_weights is not None else None
        self.criterion = nn.CrossEntropyLoss(weight=cw)

        self._val_preds = []
        self._val_true = []

        self._test_preds = []
        self._test_true = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)

        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)

        self.log("val_loss", loss, prog_bar=True, sync_dist=False)
        self._val_preds.append(preds.detach().cpu().numpy())
        self._val_true.append(y.detach().cpu().numpy())

    def on_validation_epoch_end(self):
        y_pred = np.concatenate(self._val_preds) if len(self._val_preds) else np.array([])
        y_true = np.concatenate(self._val_true) if len(self._val_true) else np.array([])
        self._val_preds.clear()
        self._val_true.clear()

        if y_true.size == 0:
            return

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
        per_class_recall = cm.diagonal() / np.clip(cm.sum(axis=1), 1, None)
        acc = (y_pred == y_true).mean()

        self.log("val_acc", float(acc), prog_bar=True)
        # optional: log mean per-class recall too
        self.log("val_mean_per_class_recall", float(per_class_recall.mean()), prog_bar=False)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        preds = logits.argmax(dim=1)
        self._test_preds.append(preds.detach().cpu().numpy())
        self._test_true.append(y.detach().cpu().numpy())

    def on_test_epoch_end(self):
        y_pred = np.concatenate(self._test_preds) if len(self._test_preds) else np.array([])
        y_true = np.concatenate(self._test_true) if len(self._test_true) else np.array([])
        self._test_preds.clear()
        self._test_true.clear()

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
        per_class_recall = cm.diagonal() / np.clip(cm.sum(axis=1), 1, None)
        acc = (y_pred == y_true).mean() if y_true.size else 0.0

        print("\n=== TEST RESULTS (Lightning) ===")
        print("Test accuracy:", round(float(acc), 4))
        print("Per-class accuracy (recall) %:", np.round(per_class_recall * 100, 2).tolist())
        print("\nClassification report (TEST):")
        print(classification_report(y_true, y_pred, digits=4))
        print("\nConfusion matrix (TEST):\n", cm)

        # also log for logger
        self.log("test_acc", float(acc))

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay
        )

        sched_type = self.hparams.scheduler

        if sched_type == "none":
            return opt

        if sched_type == "cosine":
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.trainer.max_epochs)
            return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "epoch"}}

        if sched_type == "step":
            sched = torch.optim.lr_scheduler.StepLR(opt, step_size=self.hparams.step_size, gamma=self.hparams.gamma)
            return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "epoch"}}

        if sched_type == "plateau":
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                opt, mode="min", factor=self.hparams.plateau_factor, patience=self.hparams.plateau_patience
            )
            return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "monitor": "val_loss", "interval": "epoch"}}

        raise ValueError(f"Unknown scheduler: {sched_type}")


# -----------------------------
# Trainer builder
# -----------------------------
def build_trainer(run_name: str):
    logger = TensorBoardLogger(save_dir="tb_logs", name="aerospace_rf", version=run_name)

    ckpt = ModelCheckpoint(
        monitor="val_acc",
        mode="max",
        save_top_k=1,
        filename="{epoch:02d}-{val_acc:.4f}",
        save_last=True,
    )

    early = EarlyStopping(
        monitor="val_acc",
        mode="max",
        patience=4,
        verbose=True,
    )

    lrmon = LearningRateMonitor(logging_interval="epoch")

    trainer = L.Trainer(
        max_epochs=EPOCHS,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        log_every_n_steps=25,
        callbacks=[ckpt, early, lrmon],
        logger=logger,
    )
    return trainer


# -----------------------------
# Patch size sweep runner
# -----------------------------
def run_patch_sweep(scheduler: str = "cosine"):
    L.seed_everything(RANDOM_SEED, workers=True)

    dm = PSDDataModule()
    dm.setup("fit")

    # move class weights to correct device inside module init
    cw = dm.class_weights

    patch_models = [
        "vit_base_patch8_224",
        "vit_base_patch16_224",
        "vit_base_patch32_224",
    ]

    best_overall = (-1.0, None)

    for model_name in patch_models:
        run_name = f"{model_name}_sched-{scheduler}_lr-{LR}"
        print(f"\n=== RUN: {run_name} ===")

        model = LitViT(
            model_name=model_name,
            num_classes=NUM_CLASSES,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            scheduler=scheduler,
            class_weights=cw,
        )

        trainer = build_trainer(run_name)
        trainer.fit(model, datamodule=dm)

        # Load best checkpoint for test
        best_path = trainer.checkpoint_callback.best_model_path
        if best_path:
            model = LitViT.load_from_checkpoint(
                best_path,
                model_name=model_name,
                num_classes=NUM_CLASSES,
                lr=LR,
                weight_decay=WEIGHT_DECAY,
                scheduler=scheduler,
                class_weights=cw,
            )

        trainer.test(model, datamodule=dm)

        # pull val_acc from checkpoint callback
        score = trainer.checkpoint_callback.best_model_score
        if score is not None and float(score) > best_overall[0]:
            best_overall = (float(score), run_name)

    print("\n=== BEST RUN (by val_acc) ===")
    print(best_overall)


if __name__ == "__main__":
    # options: "none", "cosine", "step", "plateau"
    run_patch_sweep(scheduler="cosine")

/usr/local/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Seed set to 42



=== RUN: vit_base_patch8_224_sched-cosine_lr-0.0003 ===


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
2026-03-03 13:30:59.032208: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772562659.196758     748 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772562659.240607     748 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772562659.686098     748 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772562659.686123     748 compu

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/usr/local/lib/python3.12/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/usr/local/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]